# Pre-encoding a dataset for DALLE·mini

This notebook shows how to pre-encode images to token sequences using JAX, VQGAN and a dataset in the [`webdataset` format](https://webdataset.github.io/webdataset/).

Adapt it to your own dataset and image encoder.

At the end you should have a dataset of pairs:
* a caption defined as a string
* an encoded image defined as a list of int.

In [1]:
from tqdm.notebook import tqdm
import torchvision.transforms as T

import webdataset as wds

import jax
import braceexpand
from pathlib import Path

## Configuration Parameters

In [2]:
shards = "/mnt/c/Users/Andrés Castro/Desktop/Datasets de Prueba/Dataset-animals-wds/dataset_shard-{000000..000001}.tar"  # defined using braceexpand format as used by webdataset
encoded_output = Path("encoded_data/Dataset-animals-parquet")  # where we will save our encoded data

IMAGE_SIZE = 256
image_transform = T.Compose([
    T.Resize(IMAGE_SIZE),          # Redimensiona el lado más corto a IMAGE_SIZE
    T.CenterCrop(IMAGE_SIZE),      # Recorta el centro a IMAGE_SIZE x IMAGE_SIZE
    T.ToTensor(),                  # Convierte la imagen PIL (H,W,C) a Tensor PyTorch (C,H,W) en [0.0, 1.0]
    T.Normalize(                   # Normaliza a [-1.0, 1.0]
        [0.5, 0.5, 0.5],           # Medias para R, G, B
        [0.5, 0.5, 0.5]            # Desviaciones estándar para R, G, B
    )
])

# VQGAN model
VQGAN_REPO = "dalle-mini/vqgan_imagenet_f16_16384"
VQGAN_COMMIT_ID = "e93a26e7707683d349bf5d5c41c5b0ef69b677a9"

# good defaults for a TPU v3-8
batch_size = 8  # Per device
num_workers = 0  # For parallel processing
total_bs = batch_size * jax.device_count()  # You can use a smaller size while testing
save_frequency = 128  # Number of batches to create a new file (180MB for f16 and 720MB for f8 per file)

In [3]:
shards = list(
    braceexpand.braceexpand(shards)
)  # better display for tqdm with known length

## Load data

We load data using `webdataset`.

In [4]:
from PIL import Image
import numpy as np

def ensure_pil(img_data):
    if isinstance(img_data, np.ndarray):
        try:
            return Image.fromarray(img_data.astype(np.uint8)) # Asumiendo formato compatible
        except: # Podrías querer un manejo de error más específico
            return img_data # O fallar explícitamente
    return img_data

ds = (
    wds.WebDataset(shards, handler=wds.warn_and_continue, shardshuffle=False, empty_check=False,)
    # .shuffle(10000, initial=1000)
    .decode("pil", handler=wds.warn_and_continue)
    .to_tuple("__key__", "jpg;png", "txt")  # assumes image is in `jpg` and caption in `txt`
    .map_tuple(None, image_transform, None) 
    .batched(total_bs)  # load in batch per worker (faster)
)

Note:
* you can also shuffle shards and items using `shardshuffle` and `shuffle` if necessary.
* you may need to resize images in your pipeline (with `map_dict` for example), we assume they are already set to 256x256.
* you can also filter out some items using `select`.

We can now inspect our data.

In [5]:
%%time
keys, images, captions = next(iter(ds))
print(keys[:50])

['0026d23740c5919f', '0055b74192ea3ab6', '00943cd3a847df1c', '009ebd2e28a2f9da', '00b0189bd5e965cb', '00b0849215c716bd', '00b34a3601c1398a', '016830e678a6d11e']
CPU times: user 363 ms, sys: 32.1 ms, total: 396 ms
Wall time: 209 ms


In [6]:
images.shape

torch.Size([8, 3, 256, 256])

In [7]:
captions[:10]

["An underwater scene featuring a fish named 'lionfish'. The fish is underwater, surrounded by moss-covered rocks. The fish itself is striped with different shades of orange and white. It has long claws on its body. There is a black eye on the fish's",
 'An underwater environment, coral reef filled with various types of marine creatures such as corals and fish. The coral reef consists of large rocks covered in pink and white coral reefs. There is a large coral reef in the middle of the image. Some of the corals have',
 'A underwater scene where a blue and white snakefish floats amongst coral reefs in an underwater environment, close-up view of its movement as it moves through the surface of the coral reef, the image is taken from low angle, allowing the viewer to focus on the fish and',
 'A brown bear standing alone in a grassy field, its black fur glistening in the sunlight as it stands tall, its white snout protrudes from its mouth, giving off a sense of calmness and alertness, peace

In [8]:
img_tensor = images[0]
img_desnormalizada = (img_tensor * 0.5) + 0.5
pil_img = T.ToPILImage()(img_desnormalizada)

Finally we create our dataloader.

In [9]:
dl = (
    wds.WebLoader(ds, batch_size=None, num_workers=0).unbatched().batched(total_bs)
)  # avoid partial batch at the end of each worker

## Image encoder

We'll use a VQGAN trained with Taming Transformers and converted to a JAX model.

In [10]:
from vqgan_jax.modeling_flax_vqgan import VQModel
from flax.jax_utils import replicate

#vqgan = VQModel.from_pretrained("flax-community/vqgan_f16_16384")
#vqgan_params = replicate(vqgan.params)

#Load VQGAN
vqgan, vq_params = VQModel.from_pretrained(
    VQGAN_REPO, revision=VQGAN_COMMIT_ID, _do_init=False
)

vqgan_params = replicate(vq_params)

/home/kstro/dalle-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Encoding

Encoding is really simple using `shard` to automatically distribute batches across devices and `pmap`.

In [11]:
from flax.training.common_utils import shard
from functools import partial


@partial(jax.pmap, axis_name="batch")
def p_encode(batch, params):
    # Not sure if we should `replicate` params, does not seem to have any effect
    _, indices = vqgan.encode(batch, params=params)
    return indices

In [12]:
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
import glob

def get_processed_filenames(output_dir):
    processed = set()
    parquet_files = glob.glob(os.path.join(output_dir, "*.parquet"))
    for f in parquet_files:
        try:
            df = pd.read_parquet(f, columns=["filename"])
            processed.update(df["filename"].tolist())
        except Exception as e:
            print(f"Error leyendo {f}: {e}")
    return processed


def encode_dataset(dataloader, output_dir: Path, save_frequency: int):
    output_dir.mkdir(parents=True, exist_ok=True)

    processed_filenames = get_processed_filenames(output_dir)
    print(f"{len(processed_filenames)} imágenes ya procesadas, se omitirán.")
    
    all_filenames = []
    all_captions = []
    all_encoding = []

    # determinar el próximo índice de archivo parquet
    existing = sorted(glob.glob(os.path.join(output_dir, "*.parquet")))
    n_file = len(existing) + 1

    per_host = batch_size * jax.device_count()

    carry_keys, carry_caps = [], []
    carry_imgs = None  # np.ndarray o None
    
    for idx, (keys, images, captions) in enumerate(tqdm(dataloader)):
        keys = list(keys)
        captions = list(captions)
        
        # Filtrar imágenes ya procesadas
        mask = [k not in processed_filenames for k in keys]
        if not any(mask):
            continue

        # Filtramos solo las no procesadas
        keys = [k for i, k in enumerate(keys) if mask[i]]
        captions = [c for i, c in enumerate(captions) if mask[i]]
        images = images[mask]

        images = images.numpy().transpose(0, 2, 3, 1).astype(np.float32)
        
        per_host = batch_size * jax.device_count()
        # 1) Acumula sobrantes
        carry_imgs = images if carry_imgs is None else np.concatenate([carry_imgs, images], axis=0)
        carry_keys += keys
        carry_caps += captions
        
        # 2) Procesa solo múltiplos de per_host
        n = (len(carry_keys) // per_host) * per_host
        if n == 0:
            continue  # espera al siguiente batch para completar
        
        proc_imgs = carry_imgs[:n]
        proc_keys = carry_keys[:n]
        proc_caps = carry_caps[:n]
        
        # 3) Mantén el remanente en carry
        carry_imgs = carry_imgs[n:]
        carry_keys = carry_keys[n:]
        carry_caps = carry_caps[n:]
        
        # 4) Codifica el bloque procesable
        proc_imgs = shard(proc_imgs)
        encoded = p_encode(proc_imgs, vqgan_params)
        encoded = np.asarray(encoded).reshape(-1, encoded.shape[-1])

        all_filenames.extend(proc_keys)
        all_captions.extend(proc_caps)
        all_encoding.extend(encoded.tolist())

        if (idx + 1) % save_frequency == 0:
            print(f"Saving file {n_file}")
            batch_df = pd.DataFrame.from_dict(
                {"filename": all_filenames, "caption": all_captions, "encoding": all_encoding}
            )
            batch_df.to_parquet(output_dir / f"{n_file:03d}.parquet")
            all_filenames = []
            all_captions = []
            all_encoding = []
            n_file += 1
    
    if carry_keys:
        m = len(carry_keys)
        pad = (per_host - m)
        pad_imgs = carry_imgs if pad == 0 else np.concatenate([carry_imgs, carry_imgs[:pad]], axis=0)
        pad_imgs = shard(pad_imgs)
        enc_pad = p_encode(pad_imgs, vqgan_params)
        enc_pad = np.asarray(enc_pad).reshape(-1, enc_pad.shape[-1])[:m]
    
        # Añade a tus buffers finales (sin split)
        all_filenames.extend(carry_keys)
        all_captions.extend(carry_caps)
        all_encoding.extend(enc_pad.tolist())
    
        # Limpia carry
        carry_imgs = None
        carry_keys, carry_caps = [], []
        
    if len(all_captions):
        print(f"Saving final file {n_file}")
        batch_df = pd.DataFrame.from_dict(
            {"filename": all_filenames, "caption": all_captions, "encoding": all_encoding}
        )
        batch_df.to_parquet(output_dir / f"{n_file:03d}.parquet")

In [13]:
encode_dataset(dl, output_dir=encoded_output, save_frequency=save_frequency)

0 imágenes ya procesadas, se omitirán.


128it [00:50,  2.86it/s]

Saving file 1


171it [01:03,  2.68it/s]


Saving final file 2


----

In [14]:
import matplotlib.pyplot as plt
from PIL import Image
import glob
import os
import pandas as pd

output_dir = encoded_output  # o el que estés usando

files = sorted(glob.glob(os.path.join(output_dir, "*.parquet")))

total = 0
for f in files:
    df = pd.read_parquet(f)
    print(f"{os.path.basename(f)}: {len(df)} imágenes")
    total += len(df)

print(f"\nTOTAL procesadas: {total}")

001.parquet: 1024 imágenes
002.parquet: 338 imágenes

TOTAL procesadas: 1362


In [15]:
import glob, os, pandas as pd

files = sorted(glob.glob("encoded_data/train_1025/*.parquet"))  # ajusta la ruta
dfs = [pd.read_parquet(f, columns=["filename"]) for f in files]
all_fns = pd.concat(dfs)["filename"].astype(str)
print("Total filas:", len(all_fns))
print("Únicos:", all_fns.nunique())
print("Duplicados:", len(all_fns) - all_fns.nunique())

ValueError: No objects to concatenate

In [ ]:
import pandas as pd
import webdataset as wds
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from pathlib import Path

# 1) Toma un parquet y construye {key -> caption}
parquet_path = sorted(Path("encoded_data").glob("*.parquet"))[0]
df = pd.read_parquet(parquet_path)[["filename", "caption"]]

# Asegura strings (por si 'filename' es numérico)
wanted = dict(zip(df["filename"].astype(str), df["caption"].astype(str)))
wanted_keys = set(wanted)  # equivalente a set(wanted.keys())

# 2) Define el shard y el resize
shard_path = "/home/kstro/proyectos/dalle-mini/dalle-mini-main/data/synth1m/data/data-000000.tar"

def resize_256_center_crop(img, size=256):
    if img.mode != "RGB":
        img = img.convert("RGB")
    return ImageOps.fit(img, (size, size), method=Image.BICUBIC)

# 3) WebDataset: decodificar antes, renombrar después
ds = (
    wds.WebDataset(
        [shard_path],
        empty_check=False,
        handler=wds.handlers.warn_and_continue,
        shardshuffle=False,
    )
    .select(lambda s: s.get("__key__", "") in wanted_keys)
    .decode("pil")                 # primero decodifica segun extension real
    .rename(image="jpg;jpeg;png")  # luego normaliza a 'image'
    .map_dict(image=resize_256_center_crop)
    .to_tuple("__key__", "image")
)

# 4) Recolecta y muestra
found = []
for key, img in ds:
    found.append((key, img, wanted[key]))
    if len(found) >= 12:  # muestra 12
        break

if not found:
    print("No se encontraron coincidencias. Verifica que 'filename' en Parquet coincide con '__key__' del shard.")
else:
    cols = 4
    rows = (len(found) + cols - 1) // cols
    plt.figure(figsize=(4*cols, 4*rows))
    for i, (_, im, cap) in enumerate(found, 1):
        plt.subplot(rows, cols, i)
        plt.imshow(im)
        plt.title(cap[:60])
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Mostradas: {len(found)}")

In [ ]:
# Verifica columnas, tipos y longitud fija de 'encoding' en tus Parquet
from datasets import load_dataset
import numpy as np

# 1) Ruta a tus parquet
data_glob = "/home/kstro/proyectos/dalle-mini/dalle-mini-main/tools/dataset/encoded_data/train_1025/*.parquet"  # EJ: r"C:\data\encoded_data\*.parquet" o "/home/usuario/encoded_data/*.parquet"

# 2) Carga (no streaming para poder inspeccionar)
ds = load_dataset("parquet", data_files={"train": [data_glob]}, streaming=False)["train"]

print("Columnas:", ds.column_names)
print("Features:", ds.features)

# 3) Validaciones básicas
assert "filename" in ds.column_names, "'filename' falta"
assert "caption" in ds.column_names, "'caption' falta"
assert "encoding" in ds.column_names, "'encoding' falta"

# 4) Asegura que encoding es lista de enteros y longitud fija
N = min(1000, len(ds))  # muestrea como mucho 1000 filas
subset = ds.select(range(N))

# Chequea que el primer elemento sea int
first_enc = subset[0]["encoding"]
assert isinstance(first_enc, (list, tuple)), f"encoding no es lista: {type(first_enc)}"
assert isinstance(first_enc[0], (int, np.integer)), f"elementos de encoding no son enteros: {type(first_enc[0])}"

# Longitudes
lens = subset.map(lambda x: {"len": len(x["encoding"])})["len"]
uniq = set(lens)
print("Longitudes únicas de encoding en la muestra:", uniq)
assert len(uniq) == 1, f"Longitudes mixtas en encoding: {uniq}"
print("OK: esquema compatible con entrenamiento.")

In [ ]:
from datasets import load_dataset
import numpy as np, pyarrow as pa

ds = load_dataset("parquet", data_files={"train": ["/home/kstro/proyectos/dalle-mini/dalle-mini-main/tools/dataset/encoded_data/train_1025/*.parquet"]}, streaming=False)["train"]
subset = ds.select(range(min(10000, len(ds))))

# Rango de códigos
mins = subset.map(lambda x: {"mn": min(x["encoding"])} )["mn"]
maxs = subset.map(lambda x: {"mx": max(x["encoding"])} )["mx"]
print("min:", min(mins), "max:", max(maxs))  # esperado: 0 <= min, max <= 16383

# Convertir a int32 (opcional, para ahorro)
# ds = ds.cast_column("encoding", pa.list_(pa.int32()))

In [ ]:
# Verificador de dataset para entrenamiento DALL·E Mini
from datasets import load_dataset, Sequence, Value
from pathlib import Path
import numpy as np
import pyarrow as pa

# 1) Directorio base donde están TODOS tus .parquet (recursivo)
BASE_DIR = "/home/kstro/proyectos/dalle-mini/dalle-mini-main/tools/dataset/encoded_data/train_1025"  # <-- CAMBIA ESTO

# 2) Recolecta paths
parquet_paths = sorted(str(p) for p in Path(BASE_DIR).rglob("*.parquet"))
assert parquet_paths, f"No se encontraron .parquet en {BASE_DIR}"
print(f"Encontrados {len(parquet_paths)} parquet(s). Ejemplo:\n  {parquet_paths[:3]}")

# 3) Carga como dataset (no streaming para inspección)
ds = load_dataset("parquet", data_files={"train": parquet_paths}, streaming=False)["train"]

if ds.features["encoding"].feature.dtype == "int64":
    print("Casting encoding -> int32")
    ds = ds.cast_column("encoding", Sequence(Value("int32")))

# 4) Chequeos de esquema
required = {"filename","caption","encoding"}
missing = required - set(ds.column_names)
assert not missing, f"Faltan columnas: {missing}"

# 5) Opcional: castear encoding a int32 (ahorra memoria)
if ds.features["encoding"].feature.dtype == "int64":
    print("Casting encoding -> int32")
    ds = ds.cast_column("encoding", pa.list_(pa.int32()))

# 6) Muestreo (o todo si es pequeño)
N = min(50000, len(ds))
subset = ds.select(range(N))
print(f"Verificando {N} filas (de {len(ds)})")

# 7) Longitud fija de encoding
lens = subset.map(lambda x: {"_len": len(x["encoding"])})["_len"]
uniq_lens = set(lens)
print("Longitudes únicas de encoding:", uniq_lens)
assert len(uniq_lens) == 1, f"Longitudes mixtas: {uniq_lens}"

# 8) Tipo correcto y rango de valores
enc0 = subset[0]["encoding"]
assert isinstance(enc0, (list, tuple))
assert isinstance(enc0[0], (int, np.integer))
mins = subset.map(lambda x: {"_mn": min(x["encoding"])})["_mn"]
maxs = subset.map(lambda x: {"_mx": max(x["encoding"])})["_mx"]
mn, mx = min(mins), max(maxs)
print(f"Rango de códigos en muestra: [{mn}, {mx}]")
# Para VQGAN f16 y config mini (image_vocab_size=16391, decoder_start_token_id=16384)
# lo normal es 0 <= code <= 16383
assert mn >= 0 and mx <= 16383, f"Códigos fuera de rango esperado [0,16383]: mn={mn}, mx={mx}"

# 9) Captions vacíos/NaN
def _is_blank(s):
    return (s is None) or (isinstance(s, float) and np.isnan(s)) or (str(s).strip() == "")
blank_flags = subset.map(lambda x: {"_blank": _is_blank(x["caption"])})["_blank"]
n_blank = sum(blank_flags)
print(f"Captions vacíos/NaN en muestra: {n_blank}/{N}")

# 10) Filenames duplicados
fn_sample = subset["filename"]
dup_count = len(fn_sample) - len(set(fn_sample))
print(f"Duplicados de filename en muestra: {dup_count}")

print("OK: verificación terminada.")